In [ ]:
from rdkit.Chem.rdMolDescriptors import CalcMolFormula
from rdkit.Chem import Draw
from src.analysis.processing import shap_ranking, shapiq_ranking
import os
import pickle
from shapiq.interaction_values import InteractionValues
from shapiq.plot.utils import format_labels

real_dataset = "../results/real_data/data_batteries_ecfp_descriptor/explanations/"
real_models = "../results/real_data/data_batteries_ecfp_descriptor"
shap_results = 'shap_results.pickle'
shapiq_results = 'shapiq1_results.pickle'

with open(os.path.join(real_dataset, shap_results), 'rb') as f:
    shap_results = pickle.load(f)
with open(os.path.join(real_dataset, shapiq_results), 'rb') as f:
    shapiq_results = pickle.load(f)

target = 'capacity_max'
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from matplotlib.gridspec import GridSpec
from rdkit import Chem

def plot_shap(shap_values, features_names, ax):
    top_10_features = np.abs(shap_values).argsort()[-10:][::-1]
    features_names = features_names[top_10_features]
    shap_values = shap_values[top_10_features]
    sns.barplot(y=features_names, x=shap_values, orient='h', ax=ax)

def plot_shapiq(shapiq_values, features_names, ax):
    top_10_features = np.abs(shapiq_values).argsort()[-10:][::-1]
    features_names = features_names[top_10_features]
    shapiq_values = shapiq_values[top_10_features]
    sns.barplot(y=features_names, x=shapiq_values, orient='h', ax=ax)

def get_features_rf(model_path, feature_names):
    rf_model = joblib.load(model_path)
    used_features = set()
    for tree in rf_model.estimators_:
        # Get the feature indices used in the current tree's splits
        used_in_tree = [i for i in tree.tree_.feature if i != -2]
        used_features.update(used_in_tree)
    # Map indices back to feature names
    used_feature_names = [feature_names[i] for i in used_features]
    return used_feature_names

import pandas as pd
import matplotlib as mpl

features_names = shap_results['test_data'][0].drop(columns=[target]).columns
feature_mapping = dict(enumerate(features_names))
shap_values = shap_results['shap_values']
shapiq_values = shapiq_results['interactions']
ranking_per_fold = []


feature_names_mapping = {
    'radius': 'Radius',
    'diameter': 'Diameter',
    'num_heteroatoms': '#Heteroatoms',
    'num_rotatable_bonds': '#Rotatable Bonds',
    'num_h_acceptors': '#H-bond Acceptors',
    'num_h_donors': '#H-bond Donors',
    'tpsa': 'TPSA',
    'mol_wt': 'Molecular Weight',
    'o%': 'O%',
    'n%': 'N%',
    'c%': 'C%'
}
for f_name in features_names:
    if f_name not in feature_names_mapping:
        f_n = f_name.split('_')[-1]
        feature_names_mapping[f_name] = f"ECFP {f_n}"

selected_examples = [
(1, 5),
(0, 0),
(1, 2),
]

for i in range(len(shap_values)):
    for j, (sv, iv_dict) in enumerate(zip(shap_values[i], shapiq_values[i])):
        if (i, j) not in selected_examples:
            continue
        print(i, j)
        smiles = shap_results['smiles'][i].iloc[j]
        print(f"Processing Fold {i}, Example {j}, SMILES: {smiles}")

        iv = InteractionValues.from_dict(iv_dict)
        interaction_list = iv.interaction_lookup.keys()

        feature_labels = np.array([format_labels(feature_tuple=iv, feature_mapping=feature_mapping) for iv in interaction_list])
        iv_ranking = pd.DataFrame({'features': feature_labels, 'ranking': iv.values})
        iv_ranking = iv_ranking[~iv_ranking['features'].str.contains('Base Value')]

        for f in features_names:
            if f not in iv_ranking['features'].values:
                iv_ranking = pd.concat([iv_ranking, pd.DataFrame({'features': [f], 'ranking': [0.]})], ignore_index=True)

        shap_series = pd.Series(sv, index=features_names)
        shapiq_series = pd.Series(iv_ranking['ranking'].values, index=iv_ranking['features'].values)

        shap_rank = shap_series.abs().rank(method='min', ascending=False).astype(int)
        shapiq_rank = shapiq_series.abs().rank(method='min', ascending=False).astype(int)
        shap_rank_10 = shap_rank.nsmallest(10).index
        shapiq_rank_10 = shapiq_rank.nsmallest(10).index
        all_features = shap_rank_10.union(shapiq_rank_10)

        #all_features = shap_series.index.union(shapiq_series.index)

        data_df = pd.DataFrame({
            'Feature': [feature_names_mapping.get(f, f) for f in all_features],
            'SHAP Value': [f for f in shap_series[all_features]],
            'SHAP-IQ Value': [f for f in shapiq_series[all_features]],
            'SHAP Rank': [f for f in shap_rank[all_features]],
            'SHAP-IQ Rank': [f for f in shapiq_rank[all_features]],
        }).reset_index(drop=True)

        top_shap_features_df = data_df.sort_values(by="SHAP Rank", ascending=True)#.head(10)

        plot_data = top_shap_features_df.sort_values(by="SHAP Rank", ascending=True)

        # 3. Create the figure.
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6), sharex=True)
        plot_data['Sign SHAP'] = np.sign(plot_data['SHAP Value'])
        sns.barplot(data=plot_data, y='Feature', x='SHAP Value', orient='h', ax=ax1,
                    hue='Sign SHAP', palette={-1: '#1e88e5', 1: '#ff0d57'}, legend=False)

        ax1.set_title('SHAP', fontsize=18)
        ax1.set_xlabel('SHAP Value', fontsize=18)
        ax1.set_ylabel('')
        ax1.axvline(0, color='k', linestyle='--', linewidth=0.8)

        def pad_sequence_right(seq, target_length, pad_value=' '):
            return seq + pad_value * (target_length - len(seq))
        def pad_sequence_left(seq, target_length, pad_value=' '):
            return pad_value * (target_length - len(seq)) + seq


        max_feature_len = plot_data['Feature'].str.len().max()
        max_shapiq_rank_len = max(len(f"[Rank: {r}]") for r in plot_data['SHAP-IQ Rank'])
        y_labels_shap = []
        for _, row in plot_data.iterrows():
            feature_text = pad_sequence_right(row['Feature'], max_feature_len)
            rank_text = f"[Rank: {row['SHAP Rank']}]"
            rank_text = pad_sequence_left(rank_text, max_shapiq_rank_len)
            y_labels_shap.append(f"{feature_text} {rank_text}")

        #y_labels_shap = [f"{row['Feature']}  [Rank: {row['SHAP Rank']}]" for _, row in plot_data.iterrows()]
        ax1.set_yticklabels(y_labels_shap, fontdict={'fontsize': 18}, fontfamily='monospace')
        ax1.tick_params(axis='y', length=0, pad=10)

        plot_data['Sign SHAP-IQ'] = np.sign(plot_data['SHAP-IQ Value'])
        sns.barplot(data=plot_data, y='Feature', x='SHAP-IQ Value', orient='h', ax=ax2,
                    hue='Sign SHAP-IQ', palette={-1: '#1e88e5', 1: '#ff0d57'}, legend=False)

        ax2.set_title('SHAP-IQ', fontsize=18)
        ax2.set_xlabel('SHAP-IQ Value', fontsize=18)
        ax2.set_ylabel('')
        ax2.axvline(0, color='k', linestyle='--', linewidth=0.8)

        ax2.yaxis.tick_right()

        ax2.set_yticklabels([]) # Hide the default feature names on the right
        for ii, (_, row) in enumerate(plot_data.iterrows()):
            rank_text = f"[Rank: {row['SHAP-IQ Rank']}]"
            rank_text = f"{pad_sequence_right(rank_text, max_shapiq_rank_len)}  {row['Feature']}"
            ax2.text(1.02, ii, rank_text,
                     transform=ax2.get_yaxis_transform(),
                     ha="left", va="center", fontsize=18, fontfamily='monospace')

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(f'../results/smiles+{i}_{j}.pdf', dpi=300, bbox_inches='tight')
        plt.show()
        mol = Chem.MolFromSmiles(smiles)
        molecular_formula = CalcMolFormula(mol)
        print(i, j, Chem.MolToSmiles(mol), molecular_formula)
        img = Draw.MolToImage(Chem.MolFromSmiles(smiles), size=(300, 300))
        fig, ax = plt.subplots(1, 1, figsize=(3, 3))
        ax.imshow(img)
        ax.axis('off')
        plt.tight_layout()
        plt.savefig(f'../results/smiles+mol_{i}_{j}.pdf', dpi=300, bbox_inches='tight')
        plt.show()